# 05 – Seleção de Features e Engenharia de Variáveis
Este notebook limpa, traduz e enriquece a base de dados para garantir o melhor desempenho e interpretabilidade nos modelos preditivos.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Paths
ROOT = Path("..").resolve()
DATA_PATH = ROOT / 'data' / 'processed' / 'base_modelagem.csv'
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'base_modelagem_reduzida.csv'

# Carregamento da base
df = pd.read_csv(DATA_PATH, low_memory=False)
df['indicador_obito'] = pd.to_numeric(df['indicador_obito'], errors='coerce')

print(f"Base carregada com {df.shape[0]} linhas e {df.shape[1]} colunas.")

Base carregada com 415367 linhas e 658 colunas.


## Etapa 1: Feature Engineering (Criação de Variáveis de Negócio)
Criação de índices combinados e lógicas clínicas para facilitar o aprendizado do modelo.

In [2]:
print("Criando novas features clínicas e estruturais...")

# 1. Escore de Fragilidade
colunas_diag_sec = ['tipo_diag_sec_1', 'tipo_diag_sec_2_cod', 'tipo_diag_sec_3_cod', 'tipo_diag_sec_4_cod']
colunas_presentes = [c for c in colunas_diag_sec if c in df.columns]
df_diag = df[colunas_presentes].replace(['0', '0.0', 0, ''], np.nan)
df['escore_fragilidade'] = df_diag.notna().sum(axis=1)

# 2. Volume Hospitalar Anual
if 'codigo_cnes' in df.columns and 'ano_competencia' in df.columns:
    df['volume_hospitalar_anual'] = df.groupby(['codigo_cnes', 'ano_competencia'])['codigo_cnes'].transform('count')

# 3. Razão de Leitos UTI
if all(c in df.columns for c in ['qtd_leitos_clinicos', 'qtd_leitos_cirurgicos', 'qtd_leitos_complementares']):
    total_leitos = df['qtd_leitos_clinicos'] + df['qtd_leitos_cirurgicos'] + df['qtd_leitos_complementares']
    df['razao_leitos_uti'] = df['qtd_leitos_complementares'] / (total_leitos + 1e-9)

# 4. Índice de Prontidão Cirúrgica
if 'possui_urgencia_emergencia' in df.columns and 'possui_centro_cirurgico' in df.columns:
    # Converter de boolean para numérico caso estejam como string/bool
    df['indice_prontidao_cirurgica'] = df['possui_urgencia_emergencia'].astype(float) + df['possui_centro_cirurgico'].astype(float)

# 5. Faixa Etária de Risco
if 'idade' in df.columns:
    bins_idade = [17, 59, 74, 84, 150]
    labels_idade = ['Adulto', 'Idoso_Jovem', 'Idoso_Medio', 'Super_Idoso']
    df['faixa_etaria_risco'] = pd.cut(df['idade'], bins=bins_idade, labels=labels_idade)
    # Força a conversão para texto para que o Dummies no NB06 funcione
    df['faixa_etaria_risco'] = df['faixa_etaria_risco'].astype(str)

print("Features criadas com sucesso!")

Criando novas features clínicas e estruturais...
Features criadas com sucesso!


## Etapa 2: Tradução de Códigos Médicos e Nomes (SHAP)
Mapeamento de códigos SIGTAP e CID-10 para descrições textuais interpretáveis, e renomeação de colunas CNES.

In [3]:
print("Traduzindo códigos SIGTAP e CID-10...")

# Dicionário de Procedimentos (SIGTAP - Cardiologia)
map_procedimentos = {
    '303060190': 'Tratamento IAM Clínico',
    '406030049': 'Angioplastia c/ Stent',
    '406030030': 'Angioplastia c/ 2 Stents',
    '406030022': 'Angioplastia (Sem Stent)',
    '406030073': 'Angioplastia Primária',
    '406030014': 'Angioplastia Coronariana (Geral)',
    '415020034': 'Intervenção Endovascular / Marcapasso',
    '211020044': 'Cateterismo Cardíaco',
    '406010536': 'Revascularização (Ponte de Safena)',
    '406010544': 'Revascularização (2+ enxertos)',
    '303060280': 'Tratamento D. Isquêmica',
    '303060107': 'Tratamento Insuf. Cardíaca',
    '406011192': 'Trombólise (Fibrinolítico)'
}

# Dicionário de Diagnósticos (CID-10 comum em Cardiologia)
map_cid = {
    'I10': 'Hipertensão',
    'E11': 'Diabetes Tipo 2',
    'E14': 'Diabetes NE',
    'I25': 'Doença Isquêmica Crônica',
    'I50': 'Insuf. Cardíaca',
    'J44': 'DPOC',
    'N18': 'Insuf. Renal Crônica',
    'I21': 'Infarto Agudo do Miocárdio',
    'I22': 'Infarto Recorrente',
    'I48': 'Fibrilação Atrial',
    'NAN': 'Nenhuma',
    'PREEXISTENTE': 'Preexistente'
}

# --- NOVO: Tradução dos Nomes das Colunas CNES ---
# Mapeamento educado para as principais variáveis do seu SHAP. 
# (Fique à vontade para ajustar algum termo conferindo a tabela CNES do SUS!)
rename_cols = {
    'habilitacao_802': 'hab_Alta_Complex_Cardiologia',
    'habilitacao_804': 'hab_Cirurgia_Cardiovascular',
    'habilitacao_2422': 'hab_Urgencia_Trauma_2422',
    'habilitacao_2404': 'hab_Pronto_Socorro_Cardiologia',
    'habilitacao_2411': 'hab_Rede_Urgencia_Emergencia',
    'habilitacao_1901': 'hab_UTI_Adulto_Tipo_II',
    'habilitacao_1504': 'hab_Hospital_Dia',
    'habilitacao_2301': 'hab_Transplante_Coracao',
    'habilitacao_2401': 'hab_Urgencia_Geral',
    'habilitacao_1101': 'hab_UTI_Especializada_1101',
    'complexidade_cod': 'Complexidade_Procedimento',
    'tipo_gestor': 'Tipo_Gestao',
    'especialidade_leito_cod': 'Especialidade_Leito_Clinico',
    'tipo_diag_sec_1': 'Comorbidade_1',
    'tipo_diag_sec_2_cod': 'Comorbidade_2',
    'tipo_diag_sec_3_cod': 'Comorbidade_3',
    'tipo_diag_sec_4_cod': 'Comorbidade_4',
    'carater_internacao': 'Carater_Internacao',
    'servico_119': 'servico_Fisioterapia_119',
    'servico_150': 'servico_Hemodinamica_150',
    'servico_105': 'servico_Eletrocardiograma_105',
    'servico_129': 'servico_Ressonancia_129',
    'servico_130': 'servico_Tomografia_130',
    'servico_110': 'servico_Lab_Clinico_110',
    'equip_76': 'equip_Eletrocardiografo',
    'equip_1': 'equip_Ressonancia',
    'equip_7': 'equip_Tomografo',
    'equip_13': 'equip_Desfibrilador',
    'equip_10': 'equip_Monitor_Multiparametrico',
    'qtd_instalacao_17': 'qtd_Consultorios',
    'qtd_instalacao_30': 'qtd_Salas_Cirurgia',
    'qtd_instalacao_31': 'qtd_Salas_Recuperacao',
    'qtd_instalacao_18': 'qtd_Salas_Emergencia',
    'qtd_instalacao_33': 'qtd_Salas_UTI',
    'qtd_leito_tipo_22': 'qtd_Leitos_Cardiologia'
}

# Aplicando tradução nas colunas de procedimento
for col in ['procedimento_realizado_cod', 'procedimento_solicitado']:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: str(int(x)) if pd.notnull(x) and isinstance(x, (int, float)) else str(x))
        df[col] = df[col].replace(map_procedimentos)

# Aplicando tradução nas colunas de diagnóstico secundário
diag_cols = [c for c in df.columns if c.startswith('tipo_diag_sec_')]
for col in diag_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.upper().str.strip()
        df[col] = df[col].replace(map_cid)

# Renomeando as colunas CNES para aparecerem bonitas no modelo
df.rename(columns=rename_cols, inplace=True)

print("Tradução concluída com sucesso!")


Traduzindo códigos SIGTAP e CID-10...
Tradução concluída com sucesso!


## Etapa 3: Remoção de Data Leakage e Identificadores
Remoção de variáveis que contêm informações financeiras ou temporais que ocorrem *após* a internação.

In [4]:
# Colunas que representam leakage, identificadores ou data release posterior ao evento
leakage_keywords = [
        'motivo_saida', 'data_saida', 'cid_morte', 'numero_aih',
    'numero_remessa', 'sequencial', 'cnpj_hospital', 'cep_paciente',
    'cpf_gestor', 'cnpj_mantenedora', 'sequencial_remessa', 'cep_estabelecimento',
    'cpf_cnpj_estabelecimento', 'cnes_cnpj_mantenedora', 'arquivo_origem',
    'data_internacao', 'ap01cv07', 'ap02cv07', 'ap03cv07', 'ap04cv07', 'ap05cv07', 'ap06cv07', 'ap07cv07', 'dt_atual',
    # --- Adicionadas para evitar vazamento e ruído geográfico/financeiro ---
    'valor_servicos_hospitalares', 'valor_servicos_profissionais', 'valor_total', 'valor_uti', 'valor_total_dolar', 'custo',
    'diarias_acompanhante', 'quantidade_diarias', 'diarias', 'dias_permanencia_cod', 'uti_mes_total_cod', 'tipo_uti',
    'municipio_residencia_cod', 'municipio_estabelecimento_cod', 'codigo_municipio', 'regiao_saude', 'distrito_sanitario', 'codigo_banco',
    'lavanderia', 'necroterio', 'lactario', 'procedimento_solicitado', 'mes_competencia', 'competencia_cod',
    'regra_contratual', 'centro_obstetrico', 'unidade_neonatal', 'banco_leite', 'orgao_expedidor',
    'comissao_', 'coleta_residuo', 'servico_apoio', 'gestao_ab', 'gestao_mc', 'gestao_ac', 'gestao_programa',
    'convenio_particular', 'plano_publico', 'plano_privado', 'convenio_sus', 'fluxo_clientela', 'tipo_prestador'
]

cols_to_drop_step1 = []
for col in df.columns:
    col_lower = col.lower()
    if any(keyword in col_lower for keyword in leakage_keywords):
        cols_to_drop_step1.append(col)

cols_to_drop_step1 = list(set(cols_to_drop_step1))
# Algumas colunas que devem ser removidas por match exato
exact_drop = ['competencia']
for c in exact_drop:
    if c in df.columns:
        cols_to_drop_step1.append(c)

cols_to_drop_step1 = list(set(cols_to_drop_step1))
# Garantir que indicador_obito nunca seja removido acidentalmente
if 'indicador_obito' in cols_to_drop_step1:
    cols_to_drop_step1.remove('indicador_obito')

df.drop(columns=[c for c in cols_to_drop_step1 if c in df.columns], inplace=True)
print(f"Removidas {len(cols_to_drop_step1)} colunas na Etapa 1.")
print(f"Formato atual: {df.shape}")


Removidas 128 colunas na Etapa 1.
Formato atual: (415367, 535)


## Etapa 4: Remoção de Colunas Constantes, Quasi-Constantes e Vazias
Limpeza matemática de colunas sem capacidade preditiva.

In [5]:
constantes = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
df.drop(columns=constantes, inplace=True)
print(f"Removidas {len(constantes)} colunas constantes na Etapa 2.")
print(f"Exemplos apagados: {constantes[:10]}")
print(f"Formato atual: {df.shape}")

quasi_constantes = []
for c in df.columns:
    if c != 'indicador_obito':
        top_freq = df[c].value_counts(normalize=True, dropna=False).iloc[0]
        if top_freq > 0.98:
            quasi_constantes.append(c)

df.drop(columns=quasi_constantes, inplace=True)
print(f"Removidas {len(quasi_constantes)} colunas quasi-constantes (>98%) na Etapa 3.")
print(f"Exemplos apagados: {quasi_constantes[:10]}")
print(f"Formato atual: {df.shape}")

alta_ausencia = [c for c in df.columns if df[c].isna().mean() > 0.99]
df.drop(columns=alta_ausencia, inplace=True)
print(f"Removidas {len(alta_ausencia)} colunas com ausência > 99% na Etapa 5.")
print(f"Exemplos apagados: {alta_ausencia[:10]}")

# Textos redundantes ou com cardinalidade extrema (exceto codigo_cnes, que pode ser agrupado futuramente se desejado, mas aqui será limpo caso necessário).
# Para modelagem, vamos dropar objects não convertidos
text_cols = df.select_dtypes(include=['object']).columns.tolist()
cols_text_drop = [c for c in text_cols if df[c].nunique() > 100]
if 'codigo_cnes' in cols_text_drop:
    cols_text_drop.remove('codigo_cnes') # Manter cnes para referencial, se precisar

df.drop(columns=cols_text_drop, inplace=True)
print(f"Removidas {len(cols_text_drop)} colunas de texto puras/alta cardinalidade.")
print(f"Apagadas: {cols_text_drop}")

print(f"Formato atual: {df.shape}")


Removidas 54 colunas constantes na Etapa 2.
Exemplos apagados: ['uti_mes_inicial', 'uti_mes_anterior', 'uti_mes_alta', 'uti_intermediaria_inicial', 'uti_intermediaria_anterior', 'uti_intermediaria_alta', 'uti_intermediaria_total', 'valor_sadt', 'valor_rn', 'valor_acompanhante']
Formato atual: (415367, 481)
Removidas 117 colunas quasi-constantes (>98%) na Etapa 3.
Exemplos apagados: ['codigo_idade_cod', 'nacionalidade_cod', 'indicador_homonimo', 'valor_sh_federal', 'valor_sp_federal', 'tipo_diag_sec_5_cod', 'tipo_diag_sec_6_cod', 'tipo_diag_sec_7_cod', 'tipo_diag_sec_8_cod', 'tipo_diag_sec_9_cod']
Formato atual: (415367, 364)
Removidas 0 colunas com ausência > 99% na Etapa 5.
Exemplos apagados: []
Removidas 8 colunas de texto puras/alta cardinalidade.
Apagadas: ['data_nascimento', 'procedimento_realizado_cod', 'diagnostico_secundario_1', 'municipio_gestor', 'codigo_agencia', 'conta_corrente', 'numero_alvara', 'data_expedicao_alvara_cod']
Formato atual: (415367, 356)


## Etapa 5: Redução de Cardinalidade (Top 10)
Para evitar explosão de colunas no One-Hot Encoding (Dummies), agrupamos categorias raras em 'Outros'.

In [6]:
cod_cols = [c for c in df.columns if c.endswith('_cod') or c in ['procedimento_solicitado', 'tipo_gestor']]

print(f"Tratando {len(cod_cols)} colunas categóricas...")
print(f"Variáveis afetadas: {cod_cols}\n")

for c in cod_cols:
    if c in df.columns:
        df[c] = df[c].astype(str) # Força a ser string (categoria)
        
        # Mantém as 10 mais frequentes e joga o resto para 'Outros'
        top_cats = df[c].value_counts().nlargest(10).index
        df[c] = df[c].where(df[c].isin(top_cats), 'Outros')

# Algumas outras colunas categóricas textuais devem ser garantidas como string para o get_dummies:
outras_cats = ['sexo', 'carater_internacao', 'raca_cor']
for c in outras_cats:
    if c in df.columns:
        df[c] = df[c].astype(str)

print("Tratamento concluído.")

Tratando 1 colunas categóricas...
Variáveis afetadas: ['diagnostico_principal_cod']

Tratamento concluído.


## Etapa 6: Revisão e Salvamento da Base Final

In [7]:
todas = df.columns.tolist()
habs = [c for c in todas if c.startswith('habilitacao_')]
equips = [c for c in todas if c.startswith('equip_')]
servs = [c for c in todas if c.startswith('servico_')]
cods = [c for c in todas if c.endswith('_cod') or c in ['procedimento_solicitado', 'tipo_gestor']]
outras = [c for c in todas if c not in habs + equips + servs + cods]

print(f"--- RESUMO DAS {len(todas)} COLUNAS RESTANTES ---")
print(f"🏥 Habilitações CNES (dummies naturais 0/1): {len(habs)}")
print(f"🛠 Equipamentos CNES: {len(equips)}")
print(f"🩺 Serviços CNES: {len(servs)}")
print(f"🔢 Códigos (Top 10 + 'Outros'): {len(cods)}")
print(f"👤 Outras (Clínicas, Demográficas, Desfecho): {len(outras)}\n")

# Exemplo: Se você quiser ver o nome de todas as habilitações que sobraram, descomente a linha abaixo:
# print("Habilitações restantes:", habs)

# ==================================================================
# REMOÇÃO MANUAL OPCIONAL
# Coloque aqui o nome de colunas específicas que você olhou e viu que não fazem 
# sentido preditivo para Infarto Agudo do Miocárdio (ex: município).
# ==================================================================
colunas_para_remover = [
    'idade_num', # Coluna duplicada de 'idade' criada em notebooks anteriores
]

if colunas_para_remover:
    df.drop(columns=[c for c in colunas_para_remover if c in df.columns], inplace=True)
    print(f"Removidas {len(colunas_para_remover)} colunas manualmente.")
    print(f"Formato atual: {df.shape}")
df.to_csv(PROCESSED_PATH, index=False)
print(f"\nBase super reduzida e tratada salva em: {PROCESSED_PATH}")


--- RESUMO DAS 356 COLUNAS RESTANTES ---
🏥 Habilitações CNES (dummies naturais 0/1): 113
🛠 Equipamentos CNES: 86
🩺 Serviços CNES: 54
🔢 Códigos (Top 10 + 'Outros'): 1
👤 Outras (Clínicas, Demográficas, Desfecho): 102

Removidas 1 colunas manualmente.
Formato atual: (415367, 355)

Base super reduzida e tratada salva em: /home/carolina/Documents/TCC Documentos/TCC/data/processed/base_modelagem_reduzida.csv


In [8]:
for col in df.columns:
    print(col)

ano_competencia
Especialidade_Leito_Clinico
diagnostico_principal_cod
idade
indicador_obito
motivo_autorizacao
Tipo_Gestao
codigo_cnes
Complexidade_Procedimento
Comorbidade_2
Comorbidade_3
Comorbidade_4
sexo
natureza_hospital
natureza_juridica
tipo_gestao
Carater_Internacao
tipo_financiamento
raca_cor
Comorbidade_1
nivel_atencao_ambulatorial
qtd_leitos_cirurgicos
qtd_leitos_clinicos
qtd_leitos_complementares
qtd_instalacao_01
qtd_instalacao_02
qtd_instalacao_03
qtd_instalacao_04
qtd_instalacao_05
qtd_instalacao_06
qtd_instalacao_07
qtd_instalacao_08
qtd_instalacao_09
qtd_instalacao_10
qtd_instalacao_11
qtd_instalacao_12
qtd_instalacao_13
qtd_instalacao_14
possui_urgencia_emergencia
qtd_instalacao_15
qtd_instalacao_16
qtd_Consultorios
qtd_Salas_Emergencia
qtd_instalacao_19
qtd_instalacao_20
qtd_instalacao_21
qtd_instalacao_22
qtd_instalacao_23
qtd_instalacao_24
qtd_instalacao_25
qtd_instalacao_26
qtd_instalacao_27
qtd_instalacao_28
qtd_instalacao_29
qtd_Salas_Cirurgia
possui_atendimento